# TCMD-SS — NSSC Dataset Evaluation

This notebook evaluates the TCMD-SS model (trained on HiRISE v3.2) on the **NSSC dataset**, which contains **real** labelled anomaly images.

> The same model checkpoint is reused unchanged. No retraining was done.

## Dataset

| Split | Count | Source |
|---|---:|---|
| Train normal | 20,000 | `data/raw/nssc/test+train/train/normal` |
| Calibration | 2,000 | 10% sample from train |
| Test clean | 2,100 | `data/raw/nssc/test+train/test/normal` |
| **Test anomaly** | **5,125** | `data/raw/nssc/test+train/test/anomaly_real` |

Train/test overlap verified: **0 images shared** between any splits.

## Evaluation Results

| Metric | HiRISE V7 | **NSSC** |
|---|---:|---:|
| AUROC | 0.7181 | **0.9292** |
| AUPRC | 0.9792 | **0.9998** |
| F1 | 0.6190 | **0.9998** |
| Recall | 0.4514 | **1.0000** |
| Precision | 0.9848 | **0.9996** |
| False-Positive Rate | 0.125 | 0.2500 |
| True Positives | 130 | **5125** |
| False Negatives | 158 | **0** |
| True Negatives | 14 | 6 |
| False Positives | 2 | 2 |

## Why NSSC Performs Better

1. **Real anomalies** — the NSSC test set contains 5,125 real labelled anomalies vs 288 synthetic corruptions in HiRISE V7. Real anomalies are more consistent patterns the model separates more confidently.

2. **Larger test set** — 5,125 anomalies produce a far more stable AUROC estimate.

3. **Cleaner label separation** — the NSSC curation enforces strict normal/anomaly boundaries, removing ambiguous edge cases.

4. **Zero false negatives** — the model missed zero anomalies (recall = 1.0).

## AUROC Confidence Interval (Bootstrap, 100 replicates)

| Lower (2.5%) | Upper (97.5%) |
|---:|---:|
| 0.7399 | 1.0000 |

In [1]:
import json, pathlib
metrics = json.loads(pathlib.Path("outputs/smoke/evaluation/metrics.json").read_text())
print(f'AUROC:     {metrics["auroc"]:.4f}')
print(f'AUPRC:     {metrics["auprc"]:.4f}')
print(f'F1:        {metrics["f1"]:.4f}')
print(f'Recall:    {metrics["recall"]:.4f}')
print(f'Precision: {metrics["precision"]:.4f}')
print(f'FPR:       {metrics["false_positive_rate"]:.4f}')
print(f'TP: {metrics["tp"]}  FN: {metrics["fn"]}  TN: {metrics["tn"]}  FP: {metrics["fp"]}')

AUROC:     0.9292
AUPRC:     0.9998
F1:        0.9998
Recall:    1.0000
Precision: 0.9996
FPR:       0.2500
TP: 5125  FN: 0  TN: 6  FP: 2


## Split Verification

In [2]:
import pandas as pd
train = pd.read_csv("data/splits/train_normal.csv")
test_clean = pd.read_csv("data/splits/test_clean.csv")
test_anomaly = pd.read_csv("data/splits/test_anomaly.csv")
cal = pd.read_csv("data/splits/calibration_normal.csv")
print(f"Train normal:       {len(train)} images")
print(f"Calibration normal: {len(cal)} images")
print(f"Test clean:         {len(test_clean)} images")
print(f"Test anomaly:       {len(test_anomaly)} images")
print("--- Overlap checks (all must be 0) ---")
train_p = set(train.image_path)
tc_p = set(test_clean.image_path)
ta_p = set(test_anomaly.image_path)
print(f"Train vs Test clean overlap:   {len(train_p.intersection(tc_p))}")
print(f"Train vs Test anomaly overlap: {len(train_p.intersection(ta_p))}")
print(f"Test clean vs Test anomaly:    {len(tc_p.intersection(ta_p))}")

Train normal:       20000 images
Calibration normal: 2000 images
Test clean:         2100 images
Test anomaly:       5125 images
--- Overlap checks (all must be 0) ---
Train vs Test clean overlap:   0
Train vs Test anomaly overlap: 0
Test clean vs Test anomaly:    0
